# v0.2.1 · Notebook 02: Splits, Spanish attacks and shortcut fixes

**LLM Security Gateway** · turns the raw sources from notebook 01 into train / validation / test splits, and fixes the shortcuts we found:

| Problem found | Fix in this notebook |
|---|---|
| Almost no Spanish attacks: a model would learn **"Spanish = safe"** | Translate ~1,500 English attacks (and matching harmless texts) with `Helsinki-NLP/opus-mt-en-es` |
| Almost every email is an attack: a model would learn **"email = attack"** | Add real emails with a *harmless* inserted sentence |
| *v0.2.1:* `validation` had **no emails**, so thresholds tuned on it blocked ~9% of harmless test emails | A group of whole emails (with their attacks, clean copies and harmless versions) goes to `validation` |
| *v0.2.1:* the baseline **memorised the 25 harmless email sentences** ("parking pass", "let me know") because train and test used the same ones | 60 harmless sentences split into **separate** train / validation / test lists, including harder ones that mention replies, instructions and AI tools |

**Rule: splits are assigned *before* anything is translated or generated.** A Spanish translation always lands in the same split as its English original, so the test set never contains a text the model has already seen in another language.

**Runtime: GPU (T4)** for the translation step. Takes about 10–15 minutes.

**Input:** `v02_sources.zip` from notebook 01 (you'll be asked to upload it).

In [ ]:
!pip -q install sentencepiece sacremoses

In [ ]:
import json, os, random, re, unicodedata, zipfile
import pandas as pd, requests, torch
from transformers import MarianMTModel, MarianTokenizer

SEED = 42
rng = random.Random(SEED)
OUT_DIR = "/content/v02"
REPO_RAW = "https://raw.githubusercontent.com/edithngalame/llm-security-gateway/main"
BIPIA_RAW = "https://raw.githubusercontent.com/microsoft/BIPIA/main/benchmark"

CFG = {
    "translate_attacks": 1500,       # English user attacks -> Spanish
    "translate_harmless": 1200,      # English harmless user texts -> Spanish (same translator, so
                                     # "sounds machine-translated" can't become a shortcut for "attack")
    "translate_email_attacks": 300,  # BIPIA emails with attacks -> Spanish
    "translate_email_benign": 300,   # emails with harmless inserts -> Spanish
    "max_chars": 600,                # translate only texts that fit comfortably in the model
    "benign_per_train_email": 20,
    "benign_per_eval_email": 10,     # harmless versions per validation / test email
    "val_email_frac": 0.2,           # share of BIPIA training emails moved to validation (v0.2.1)
    "mt_model": "Helsinki-NLP/opus-mt-en-es",
}
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: switch runtime to T4 GPU")
pd.set_option("display.max_colwidth", 120)

## Helper functions
Unit-tested before this notebook was written.

In [ ]:
# Harmless sentences inserted into real emails. They teach the model that an extra
# sentence in an email is not an attack by itself: only instructions aimed at the AI are.
# v0.2.1: three SEPARATE lists, so a model can't pass the test by recognising sentences it
# was trained on. Each list mixes plain sentences with harder ones that mention replies,
# instructions, "ignore" or AI tools while still being harmless.
BENIGN_INSERTS = {
    "train": [
        "Let me know if you have any questions.",
        "Please find the agenda attached.",
        "What time works best for you on Thursday?",
        "Could you confirm the delivery address?",
        "I'll be out of the office next Monday.",
        "Thanks again for your help with the report.",
        "Can we move our call to 3pm?",
        "The invoice number is listed below for reference.",
        "Please reply by Friday so we can finalise the order.",
        "Have a great weekend!",
        "Do you know if the parking is free at the venue?",
        "Remember to bring your ID for the security desk.",
        "The slides from yesterday's meeting are on the shared drive.",
        "Is the summary due this week or next?",
        "Feel free to forward this to anyone on your team.",
        "We updated the prices on our website last month.",
        "Our support team is available from 9am to 5pm.",
        "Could you send me the latest version of the contract?",
        "The office will be closed on the public holiday.",
        "Please ignore my previous email, the meeting is on Tuesday, not Monday.",
        "Note: the earlier instructions for the parking pass have changed, use gate B.",
        "What is the best way to reach you this afternoon?",
        "The package should arrive within three business days.",
        "I have attached the receipt for your records.",
        "Let's catch up after the conference.",
        "In your reply, please include your order number so we can find it quickly.",
        "If you forward this to your assistant, ask them to book the room for Tuesday.",
        "Please disregard the draft I sent earlier; the final version is attached.",
        "The system will be down for maintenance on Saturday night.",
        "Our new chatbot on the website can answer questions about delivery times.",
        "Reply with your preferred date and we'll confirm by email.",
        "The instructions for the new printer are pinned in the kitchen.",
        "Could you summarise the main points for the team before Friday?",
        "IT will update all laptops overnight, so please save your work.",
        "Please answer the survey by the end of the month.",
        "Respond to this thread if you'd like to join the Friday lunch.",
    ],
    "val": [
        "Could you share the photos from the team dinner?",
        "The training session has moved to room 4.",
        "When you reply, mention which workshop you want to attend.",
        "Ignore the calendar invite from this morning, it was sent by mistake.",
        "The AI tool we trialled last month will be available to everyone in June.",
        "Please send your expense forms to finance before the 25th.",
        "Our assistant manager will cover the front desk tomorrow.",
        "New instructions for booking travel are on the intranet page.",
        "Let us know your T-shirt size for the charity run.",
        "The quarterly figures look better than expected.",
        "Please keep this schedule confidential until the announcement.",
        "Your response to the client should go out before noon.",
    ],
    "test": [
        "Would Wednesday morning work for a quick call?",
        "The heating in the second floor office is fixed now.",
        "In your answer, please say whether you can attend in person.",
        "Forget about the old form, HR has published a new one.",
        "We are testing a new virtual assistant for customer questions next week.",
        "The contract renewal is due at the end of the quarter.",
        "Please follow the safety instructions posted at the entrance.",
        "Can someone from your team review the translation?",
        "Tickets for the conference are now available on the website.",
        "The previous guidelines on remote work still apply.",
        "Please respond only if you have not received the package.",
        "Congratulations on the new role!",
    ],
}


def dedup_key(text: str) -> str:
    t = unicodedata.normalize("NFKC", text).lower()
    t = re.sub(r"[^\w]+", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def stratified_split(df: pd.DataFrame, by: list[str], fracs=(0.8, 0.1, 0.1), seed=42,
                     names=("train", "val", "test")) -> pd.Series:
    """Assign each row to train/val/test, keeping the mix of `by` columns equal in every split."""
    rng = random.Random(seed)
    out = pd.Series(index=df.index, dtype=object)
    for _, g in df.groupby(by, sort=True):
        idx = list(g.index)
        rng.shuffle(idx)
        n = len(idx)
        n_val = int(round(n * fracs[1]))
        n_test = int(round(n * fracs[2]))
        if n >= 3:  # tiny groups go entirely to train
            n_val, n_test = max(n_val, 1), max(n_test, 1)
        else:
            n_val = n_test = 0
        out[idx[:n_test]] = names[2]
        out[idx[n_test:n_test + n_val]] = names[1]
        out[idx[n_test + n_val:]] = names[0]
    return out


def insert_sentence(context: str, sentence: str, rng: random.Random) -> str:
    position = rng.choice(["start", "middle", "end"])
    if position == "start":
        return f"{sentence}\n{context}"
    if position == "end":
        return f"{context}\n{sentence}"
    parts = re.split(r"(?<=[.!?])\s+", context)
    if len(parts) < 2:
        return f"{context}\n{sentence}"
    cut = rng.randint(1, len(parts) - 1)
    return " ".join(parts[:cut]) + f" {sentence} " + " ".join(parts[cut:])


def benign_insert_emails(emails: list[str], per_email: int, rng: random.Random, split: str) -> list[dict]:
    """Harmless versions of each email, using only the sentence list that belongs to `split`."""
    pool = BENIGN_INSERTS[split]
    rows = []
    for e in emails:
        for s in rng.sample(pool, k=min(per_email, len(pool))):
            rows.append({"text": insert_sentence(e, s, rng), "label": 0, "source": "retrieved", "lang": "en",
                         "family": "email_benign_insert", "origin": "microsoft/BIPIA + project benign inserts",
                         "license": "MIT", "split_hint": "pool", "split": split})
    return rows


def email_sentences(email: str) -> set[str]:
    """An email's sentences, normalised. Attacks and harmless sentences are inserted *between*
    sentences, so all of them survive inside every version built from the email."""
    return {k for k in (dedup_key(p) for p in re.split(r"(?<=[.!?])\s+", email)) if k}


def email_groups(emails: list[str]) -> tuple[list[int], dict[int, set[str]]]:
    """BIPIA repeats some emails word for word: identical emails form ONE group.
    Returns the group id of each email, and the sentences of each group."""
    ids, group_of = [], {}
    for e in emails:
        ids.append(group_of.setdefault(dedup_key(e), len(group_of)))
    sentences = {}
    for e, g in zip(emails, ids):
        sentences.setdefault(g, email_sentences(e))
    return ids, sentences


def base_email(text: str, group_sentences: dict[int, set[str]]):
    """Group of the email a text was built from: the email whose sentences ALL appear in it.
    Many emails share boilerplate sentences, so one shared sentence is not enough.
    If several match (one email contained in another), take the longest."""
    k = dedup_key(text)
    full = [g for g, ss in group_sentences.items() if all(s in k for s in ss)]
    return max(full, key=lambda g: sum(map(len, group_sentences[g]))) if full else None


def pick_val_groups(n_groups: int, frac: float, seed=42) -> set[int]:
    """Whole emails that go to validation, so every version of an email stays in one split."""
    return set(random.Random(seed).sample(range(n_groups), k=max(1, round(frac * n_groups))))


def assign_splits(df: pd.DataFrame, emails_train: list[str], emails_test: list[str],
                  val_groups: set[int], seed=42) -> pd.Series:
    """Group-aware split.

    - hand-written sets        -> eval_handwritten (never trained on)
    - other languages          -> test_other_lang
    - BIPIA held-out attacks   -> test_heldout
    - BIPIA pool attacks       -> validation if built from a validation email, else train
                                  (an email's versions must never be split across train and validation)
    - clean BIPIA emails       -> test for BIPIA test emails, validation for validation emails, else train
    - everything else          -> stratified 80/10/10 by label, source, language
    """
    split = pd.Series(index=df.index, dtype=object)
    split[df.split_hint == "eval_only"] = "eval_handwritten"
    split[df.split_hint == "other_lang"] = "test_other_lang"
    split[df.split_hint == "heldout_attack"] = "test_heldout"
    bipia = (df.origin == "microsoft/BIPIA") & (df.split_hint == "pool")
    test_emails = set(emails_test)
    _, sentences = email_groups(emails_train)
    unmatched = 0
    for idx, text in df.loc[bipia, "text"].items():
        if df.at[idx, "label"] == 0 and text in test_emails:
            split[idx] = "test"
            continue
        g = base_email(text, sentences)
        unmatched += g is None
        split[idx] = "val" if g in val_groups else "train"
    print(f"BIPIA rows whose email couldn't be identified (kept in train): {unmatched}")
    rest = split.isna()
    split[rest] = stratified_split(df[rest], ["label", "source", "lang"], seed=seed)
    return split


def email_groups_by_split(df: pd.DataFrame, emails_train: list[str]) -> pd.Series:
    """For every training email: in which splits do its versions appear? Each should be in exactly one."""
    _, sentences = email_groups(emails_train)
    en = df[df.origin.str.contains("BIPIA") & (df.lang == "en") & df.split.isin(["train", "val"])]
    base = en.text.map(lambda t: base_email(t, sentences))
    return en.assign(email=base).dropna(subset=["email"]).groupby("email").split.agg(lambda s: sorted(set(s)))


def pick_for_translation(df: pd.DataFrame, n: int, max_chars: int, seed=42) -> pd.DataFrame:
    """English user attacks to translate, spread evenly across attack families."""
    cand = df[(df.label == 1) & (df.lang == "en") & (df.source == "user")
              & (df.text.str.len() <= max_chars) & df.split.isin(["train", "val", "test"])]
    fams = cand.groupby("family")
    per_fam = max(1, n // max(1, fams.ngroups))
    picked = pd.concat([g.sample(min(len(g), per_fam), random_state=seed) for _, g in fams])
    if len(picked) < n:
        extra = cand.drop(picked.index)
        picked = pd.concat([picked, extra.sample(min(len(extra), n - len(picked)), random_state=seed)])
    return picked.head(n)


def leakage_report(df: pd.DataFrame) -> pd.DataFrame:
    """Texts (by dedup key) that appear in more than one split: should be empty."""
    k = df.assign(key=df.text.map(dedup_key))
    multi = k.groupby("key")["split"].nunique()
    bad = multi[multi > 1].index
    return k[k.key.isin(bad)].sort_values("key")[["split", "origin", "text"]]

## 1. Load the sources from notebook 01

In [ ]:
PARQUET = f"{OUT_DIR}/sources_v0.2.parquet"
if not os.path.exists(PARQUET):
    from google.colab import files
    print("Upload v02_sources.zip from notebook 01:")
    up = files.upload()
    name = next(iter(up))
    os.makedirs(OUT_DIR, exist_ok=True)
    zipfile.ZipFile(name).extractall(OUT_DIR)
df = pd.read_parquet(PARQUET)
print(f"{len(df):,} rows loaded")

## 2. Assign splits (before any translation)

*v0.2.1:* 20% of BIPIA's training emails go to `validation` **as whole emails**: the attacked versions, the clean copy and (next step) the harmless versions all land there together.

In [ ]:
def get(url):
    r = requests.get(url, timeout=60); r.raise_for_status(); return r.text

def contexts(path):
    return [json.loads(l)["context"] for l in get(f"{BIPIA_RAW}/{path}").splitlines() if l.strip()]

emails_train, emails_test = contexts("email/train.jsonl"), contexts("email/test.jsonl")
group_ids, group_sentences = email_groups(emails_train)
val_groups = pick_val_groups(len(group_sentences), CFG["val_email_frac"], seed=SEED)
print(f"BIPIA emails: {len(emails_train)} train ({len(group_sentences)} distinct, {len(val_groups)} moved to validation), "
      f"{len(emails_test)} test")
df["split"] = assign_splits(df, emails_train, emails_test, val_groups, seed=SEED)
display(pd.crosstab(df["split"], df["label"], margins=True).rename(columns={0: "harmless", 1: "attack"}))

## 3. Fix the email shortcut: real emails with a harmless inserted sentence

In [ ]:
train_only = list(dict.fromkeys(e for e, g in zip(emails_train, group_ids) if g not in val_groups))
val_only = list(dict.fromkeys(e for e, g in zip(emails_train, group_ids) if g in val_groups))
benign = (benign_insert_emails(train_only, CFG["benign_per_train_email"], rng, "train")
          + benign_insert_emails(val_only, CFG["benign_per_eval_email"], rng, "val")
          + benign_insert_emails(emails_test, CFG["benign_per_eval_email"], rng, "test"))
df = pd.concat([df, pd.DataFrame(benign)], ignore_index=True)
print(f"added {len(benign)} harmless emails")
emails = df[(df.source == "retrieved") & df.origin.str.contains("BIPIA")]
display(pd.crosstab(emails["split"], emails["label"]).rename(columns={0: "harmless", 1: "attack"}))

In [ ]:
# v0.2.1 checks: run these before spending GPU time on translation
lists = {k: set(v) for k, v in BENIGN_INSERTS.items()}
checks = {
    "harmless sentence lists don't overlap": not (lists["train"] & lists["val"] or lists["train"] & lists["test"]
                                                  or lists["val"] & lists["test"]),
    "validation has harmless emails": ((df.split == "val") & (df.label == 0) & df.text.str.contains("SUBJECT:", regex=False)).any(),
    "validation has email attacks": ((df.split == "val") & (df.label == 1) & (df.source == "retrieved")).any(),
    "every email stays in one split": email_groups_by_split(df, emails_train).map(len).max() == 1,
}
for name, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + name)
assert all(checks.values()), "fix the failing check before translating"

## 4. Fix the Spanish shortcut: translate attacks *and* matching harmless texts

Each translation keeps the split of its English original. Harmless texts go through the **same** translator, so the model can't learn "machine-translated style = attack".

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = MarianTokenizer.from_pretrained(CFG["mt_model"])
mt = MarianMTModel.from_pretrained(CFG["mt_model"]).to(device).eval()

@torch.no_grad()
def translate(texts, batch_size=32):
    """Translate with the model directly instead of the `pipeline` helper, whose output
    format changed between library versions (it started returning dicts, not text)."""
    texts, out = list(texts), []
    for i in range(0, len(texts), batch_size):
        batch = tok(texts[i:i + batch_size], return_tensors="pt", padding=True,
                    truncation=True, max_length=512).to(device)
        out += tok.batch_decode(mt.generate(**batch, max_new_tokens=512), skip_special_tokens=True)
    assert all(isinstance(t, str) for t in out), "translator returned non-text output"
    return out

def translated_rows(src: pd.DataFrame) -> pd.DataFrame:
    es = src.copy()
    es["text"] = translate(src["text"])
    es["lang"] = "es"
    es["origin"] = src["origin"] + " (MT-es: opus-mt-en-es)"
    return es

trainable = df.split.isin(["train", "val", "test"])
attacks = pick_for_translation(df, CFG["translate_attacks"], CFG["max_chars"], seed=SEED)
harmless = df[trainable & (df.label == 0) & (df.lang == "en") & (df.source == "user")
              & (df.text.str.len() <= CFG["max_chars"])].sample(CFG["translate_harmless"], random_state=SEED)
email_att = df[df.split.isin(["train", "val"]) & (df.label == 1) & (df.source == "retrieved")].sample(
    CFG["translate_email_attacks"], random_state=SEED)
email_ben = df[(df.family == "email_benign_insert")].sample(CFG["translate_email_benign"], random_state=SEED)
email_clean = df[df.family == "bipia_clean_email"]
heldout = df[df.split == "test_heldout"].sample(min(150, (df.split == "test_heldout").sum()), random_state=SEED)

parts = {"attacks": attacks, "harmless": harmless, "email attacks": email_att,
         "email harmless inserts": email_ben, "clean emails": email_clean, "held-out email attacks": heldout}
translated = []
for name, part in parts.items():
    print(f"translating {len(part):>5} {name} ...", flush=True)
    translated.append(translated_rows(part))
es = pd.concat(translated, ignore_index=True)
print(f"done: {len(es):,} Spanish rows")

In [ ]:
# Read some translations before trusting them
for _, r in es.groupby("label").sample(4, random_state=SEED).iterrows():
    print(f"[{'ATTACK' if r['label'] else 'harmless'} | {r['source']} | {r['split']}] {str(r['text'])[:220]}")

## 5. Your native Spanish test set

Loaded from `data/samples/native_es.jsonl` in your GitHub repo. **Test only, never trained on.** It measures how the model handles Spanish written by a real person rather than a translator.

In [ ]:
try:
    rows = [json.loads(l) for l in get(f"{REPO_RAW}/data/samples/native_es.jsonl").splitlines()
            if l.strip() and not l.lstrip().startswith("//")]
    native = pd.DataFrame([{**{k: r.get(k) for k in ["text", "label", "source", "lang", "family"]},
                            "origin": "handwritten/native_es.jsonl", "license": "project",
                            "split_hint": "eval_only", "split": "test_native_es"} for r in rows])
    native["lang"] = native["lang"].fillna("es"); native["source"] = native["source"].fillna("user")
    print(f"loaded {len(native)} native Spanish rows "
          f"({int(native.label.sum())} attacks, {int((native.label == 0).sum())} harmless)")
except Exception as e:
    native = pd.DataFrame()
    print(f"native_es.jsonl not found yet ({e}). Fine for now: add it to the repo and re-run this notebook.")

## 6. Combine, remove duplicates, check for leakage between splits

In [ ]:
full = pd.concat([df, es, native], ignore_index=True)
full["key"] = full["text"].map(dedup_key)
before = len(full)
full = full.drop_duplicates("key", keep="first")
print(f"removed {before - len(full)} duplicates created by translation")

leaks = leakage_report(full[full.split.isin(["train", "val", "test", "test_heldout", "test_native_es",
                                              "test_other_lang", "eval_handwritten"])])
print(f"texts found in more than one split: {leaks['text'].nunique() if len(leaks) else 0}")
assert len(leaks) == 0, "leakage between splits! inspect `leaks`"

## 7. The final dataset

In [ ]:
print(f"TOTAL: {len(full):,} rows | attacks: {(full.label == 1).sum():,} | harmless: {(full.label == 0).sum():,}\n")
display(pd.crosstab(full["split"], full["label"], margins=True).rename(columns={0: "harmless", 1: "attack"}))
print("\nTRAIN by source and language (both labels should appear in every row):")
tr = full[full.split == "train"]
display(pd.crosstab([tr["source"], tr["lang"]], tr["label"]).rename(columns={0: "harmless", 1: "attack"}))

In [ ]:
# The two shortcuts, before vs after
def share(d, mask):
    s = d[mask]; return f"{(s.label == 1).mean():.0%} attacks ({len(s):,} rows)"
print("Spanish texts in TRAIN:   ", share(tr, tr.lang == "es"))
print("Retrieved docs in TRAIN:  ", share(tr, tr.source == "retrieved"))
print("  of which email-shaped:  ", share(tr, tr.text.str.contains("SUBJECT:", regex=False)))
va = full[full.split == "val"]
print("Email-shaped in VALIDATION:", share(va, va.text.str.contains("SUBJECT:", regex=False)), " (v0.2: 0 rows)")

## 8. Save

In [ ]:
import shutil
final = full.drop(columns=["key"])
final.to_parquet(f"{OUT_DIR}/dataset_v0.2.parquet", index=False)
os.makedirs(f"{OUT_DIR}/splits", exist_ok=True)
for name, g in final.groupby("split"):
    g.to_json(f"{OUT_DIR}/splits/{name}.jsonl", orient="records", lines=True, force_ascii=False)
summary = {"version": "v0.2.1", "rows": len(final), "cfg": CFG, "seed": SEED,
           "splits": {k: {"harmless": int((g.label == 0).sum()), "attack": int((g.label == 1).sum())}
                      for k, g in final.groupby("split")}}
json.dump(summary, open(f"{OUT_DIR}/dataset_summary.json", "w"), indent=2)
shutil.make_archive("/content/v02_dataset", "zip", OUT_DIR)
print(json.dumps(summary["splits"], indent=2))
from google.colab import files
files.download("/content/v02_dataset.zip")